# Notebook 04 — Closed-Loop Dual-Track Generation (With WQB)

**Goal**: Run the full dual-track orchestration loop:
```
[Explorer]  Idea → Synthesis → Validate → Novelty(strict)  ──┐
                                                               ├─► WQB Simulate → Qualify → Reflect
[Skeleton]  SkeletonRegistry → SkeletonAgent → Validate ──────┘         │
                                                                         ▼
                                                              SkeletonExtractor → SkeletonRegistry
```

**Three `track_mode` options:**
- `"explorer_only"` — only the free-exploration LLM track (original behavior)
- `"skeleton_only"` — only the controlled skeleton-variation track
- `"hybrid"` *(default)* — both tracks, budget split by TrackBandit (UCB)

**Requires**: Notebooks 01–03 completed. WQB credentials set in `.env`.

⚠️ **WQB API quota**: each expression costs one simulation. Start with `n_rounds=1`.

In [1]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from alpha_agent.config import settings
from alpha_agent.knowledge.vector_store import VectorStore
from alpha_agent.knowledge.alpha_memory import AlphaMemory
from alpha_agent.knowledge.skeleton_registry import SkeletonRegistry
from alpha_agent.data.operator_kb import OperatorKB
from alpha_agent.data.wqb_client import WQBClient
from alpha_agent.engine.orchestrator import Orchestrator
from alpha_agent.search.bandit import DirectionBandit
from alpha_agent.search.track_bandit import TrackBandit

store    = VectorStore()
memory   = AlphaMemory()
skeleton_db_path = settings.duckdb_path.with_name("skeleton_registry.db")
registry = SkeletonRegistry(db_path=skeleton_db_path)
kb       = OperatorKB()

print("AlphaMemory DB:", settings.duckdb_path)
print("SkeletonRegistry DB:", skeleton_db_path)
print("SkeletonRegistry status:", registry.stats())

mem_status = memory.status()
print("AlphaMemory status:", mem_status)
if mem_status.get("read_only"):
    raise RuntimeError(
        "AlphaMemory is read-only in this kernel (likely DB lock by another notebook/process). "
        "Please stop other kernels using alpha_memory.db and rerun this cell."
    )

/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/myenv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


AlphaMemory DB: /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/alpha_memory.db
SkeletonRegistry DB: /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/skeleton_registry.db
SkeletonRegistry status: {'total': 11, 'active': 11, 'instances': 1}
AlphaMemory status: {'path': '/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/alpha_memory.db', 'read_only': False, 'ephemeral': False, 'lock_message': ''}


## 1. Configure the run

In [2]:
# ─── Configuration ────────────────────────────────────────────────────────
DIRECTION  = "short-term price reversal with liquidity adjustment"
DATASET    = "pv1"
UNIVERSE   = "TOP1000"
N_ROUNDS   = 2        # start small! increase after validating setup
IDEAS_PER_ROUND   = 3
VARIANTS_PER_IDEA = 2
EXPLORE_BIAS = 0.4   # 0=explore, 1=exploit
AUTO_SUBMIT  = False  # set True to auto-submit qualified alphas
DRY_RUN      = False  # True = skip WQB (for testing prompt quality)

# ─── Track mode ───────────────────────────────────────────────────────────
# "hybrid"        → TrackBandit splits budget between Explorer + Skeleton
# "explorer_only" → only free-exploration LLM (original behavior)
# "skeleton_only" → only SkeletonAgent (needs seeds in registry first)
TRACK_MODE = "hybrid"
# ──────────────────────────────────────────────────────────────────────────

## 2. (Optional) Use Bandit to auto-select direction

In [3]:
# Uncomment to let the bandit choose the direction automatically
# bandit = DirectionBandit(memory)
# DIRECTION = bandit.select()
# print(f"Bandit selected direction: {DIRECTION}")

print(f"Direction: {DIRECTION}")
print(f"Dataset:   {DATASET} / {UNIVERSE}")
print(f"Rounds:    {N_ROUNDS}")
print(f"Dry run:   {DRY_RUN}")

Direction: short-term price reversal with liquidity adjustment
Dataset:   pv1 / TOP1000
Rounds:    2
Dry run:   False


## 3. Run the orchestration loop

In [4]:
async def run_pipeline():
    async with WQBClient() as client:
        orch = Orchestrator(
            client=client,
            vector_store=store,
            alpha_memory=memory,
            skeleton_registry=registry,
            operator_kb=kb,
            auto_submit=AUTO_SUBMIT,
        )
        reports = await orch.run(
            direction=DIRECTION,
            dataset=DATASET,
            universe=UNIVERSE,
            n_rounds=N_ROUNDS,
            ideas_per_round=IDEAS_PER_ROUND,
            variants_per_idea=VARIANTS_PER_IDEA,
            explore_exploit_bias=EXPLORE_BIAS,
            dry_run=DRY_RUN,
            track_mode=TRACK_MODE,
        )
    return reports

reports = asyncio.run(run_pipeline())

───────────────────────── Round 1/2 — short-term price reversal with liquidity adjustment ─────────────────────────

Allocation — explorer=1 | skeleton=5

[Explorer] Generating 3 ideas...

[Skeleton] Generating variants from 1 seeds...

[Explorer] Synthesizing 2 variants/idea...

...

Simulating 5 expressions on WQB...

Reflecting on results...

                                Round 2 Summary                                
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                ┃                                               Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ round                 │                                                   2 │
│ direction             │ short-term price reversal with liquidity adjustment │
│ ideas                 │                                                   0 │
│ expressions_generated │                                                   5 │
│ after_validation      │                                                   5 │
│ after_novelty_filter  │                                                   5 │
│ simulated             │                                                   2 │
│ qualified             │                                                   0 │
│ soft_qualified        │                                                   0 │
│ skeletons_added       │                                                   0 │
│ pass_rate             │                                                 0.0 │
│ explorer_simulated    │                                                   0 │
│ explorer_qualified    │                                                   0 │
│ skeleton_simulated    │                                                   2 │
│ skeleton_qualified    │                                                   0 │
│ duration_s            │                                          251.136757 │
└───────────────────────┴─────────────────────────────────────────────────────┘

──────────────────────────────────────────────── Session Complete ─────────────────────────────────────────────────

Total rounds: 2 | Simulated: 6 | Qualified: 0 | Pass rate: 0.0%

## 4. Review results

In [5]:
import pandas as pd

summary_rows = [r.to_dict() for r in reports]
df_summary = pd.DataFrame(summary_rows)
print("Round-by-round summary:")
display(df_summary)

Round-by-round summary:


,round,direction,ideas,expressions_generated,after_validation,after_novelty_filter,simulated,qualified,soft_qualified,skeletons_added,pass_rate,explorer_simulated,explorer_qualified,skeleton_simulated,skeleton_qualified,duration_s
0,1,short-term price reversal with liquidity adjus...,1,6,6,6,4,0,0,0,0.0,0,0,4,0,1102.116766
1,2,short-term price reversal with liquidity adjus...,0,5,5,5,2,0,0,0,0.0,0,0,2,0,251.136757


In [6]:
# Check alpha memory status
stats = memory.stats()
print(f"Alpha Memory after run:")
print(f"  Total:     {stats['total']}")
print(f"  Qualified: {stats['qualified']}")
print(f"  Pass rate: {stats['pass_rate']:.1%}")

Alpha Memory after run:
  Total:     6
  Qualified: 0
  Pass rate: 0.0%


In [7]:
# View top qualified alphas
top = memory.top_by_metric('sharpe', k=5, qualified_only=True)
print("Top 5 qualified alphas by Sharpe:")
for a in top:
    m = a['metrics']
    print(f"  sharpe={m.get('sharpe',0):.3f} fitness={m.get('fitness',0):.3f}")
    print(f"  {a['expression'][:90]}")
    print()

Top 5 qualified alphas by Sharpe:


In [8]:
# View recent reflections (learning)
recent = memory.recent(n=10)
failed = [a for a in recent if not a['qualified'] and a['reflection']]

print("Recent failure reflections:")
for a in failed[:3]:
    print(f"\n  Expression: {a['expression'][:60]}")
    print(f"  Reflection: {str(a['reflection'])[:300]}")

Recent failure reflections:

  Expression: group_rank(ts_std_dev(open, 100), sector)
  Reflection: ```json
{
  "root_cause": "The alpha uses a slow-moving volatility measure (100-day standard deviation of open prices) combined with sector-neutral ranking, creating a signal with insufficient predictive power and excessively low turnover. The 100-day window smooths the signal too much, while sector

  Expression: group_rank(ts_std_dev(close, 50), industry)
  Reflection: ```json
{
  "root_cause": "The alpha signal uses a slow-moving volatility measure (50-day standard deviation of close prices) ranked within industries, resulting in an overly stable signal with no predictive power. The close price standard deviation is economically less meaningful than returns volat

  Expression: group_rank((returns - open) / open, subindustry)
  Reflection: ```json
{
  "root_cause": "The expression calculates a noisy intraday return signal ((returns-open)/open) and applies group ranking without any smoo

## 5. Skeleton Registry — Results

In [9]:
# Skeleton registry stats after the run
reg_stats = registry.stats()
print("Skeleton Registry Stats:")
for k, v in reg_stats.items():
    print(f"  {k}: {v}")

# Show all active skeletons
skeletons = registry.all()
print(f"\nActive skeletons: {len(skeletons)}")
for sk in skeletons[:10]:
    print(f"\n  [{sk.skeleton_id[:8]}] {sk.template_str[:80]}")
    print(f"    operators: {sk.operators_used}")
    print(f"    field_slots: {[s['name'] for s in sk.field_slots]}")
    print(f"    attempts={sk.attempt_count}  qualified={sk.success_count}  rate={sk.success_rate:.0%}")

Skeleton Registry Stats:
  total: 11
  active: 11
  instances: 7

Active skeletons: 11

  [2a6d5b37] group_rank(($X1 - $X2) / $X2, $G1)
    operators: ['group_rank']
    field_slots: ['$X1', '$X2']
    attempts=5  qualified=0  rate=0%

  [3128cf7d] group_rank(ts_std_dev($X1, $W1), $G1)
    operators: ['group_rank', 'ts_std_dev']
    field_slots: ['$X1']
    attempts=2  qualified=0  rate=0%

  [42aab681] group_rank(ts_std_dev($X1, $W1) / ts_mean($X2, $W2), $G1)
    operators: ['group_rank', 'ts_std_dev', 'ts_mean']
    field_slots: ['$X1', '$X2']
    attempts=0  qualified=0  rate=0%

  [7c625474] ts_corr($X1, abs($X2), $W1)
    operators: ['ts_corr']
    field_slots: ['$X1', '$X2']
    attempts=0  qualified=0  rate=0%

  [e7c5d916] group_rank(($X1 - ts_mean($X1, $W1)) / ts_std_dev($X1, $W1), $G1)
    operators: ['group_rank', 'ts_mean', 'ts_std_dev']
    field_slots: ['$X1']
    attempts=0  qualified=0  rate=0%

  [dfe885dd] trade_when(ts_rank(ts_std_dev($X1, $W1), $W2) < $W3, $X2, -1)


In [10]:
# TrackBandit stats
from alpha_agent.search.track_bandit import TrackBandit
bandit = TrackBandit(skeleton_registry=registry)
# Show allocation preview for different budgets
for budget in [5, 10, 20]:
    print(bandit.allocation_preview(budget))

# Bandit stats from reports
print("\nPer-round track performance:")
import pandas as pd
track_rows = []
for r in reports:
    track_rows.append({
        "round": r.round_num,
        "explorer_sim": r.explorer_simulated,
        "explorer_qual": r.explorer_qualified,
        "skeleton_sim": r.skeleton_simulated,
        "skeleton_qual": r.skeleton_qualified,
        "skeletons_added": r.skeletons_added,
    })
df_tracks = pd.DataFrame(track_rows)
display(df_tracks)

Budget=5: explorer=1 (20%) | skeleton=4 (80%)
Budget=10: explorer=3 (30%) | skeleton=7 (70%)
Budget=20: explorer=6 (30%) | skeleton=14 (70%)

Per-round track performance:


,round,explorer_sim,explorer_qual,skeleton_sim,skeleton_qual,skeletons_added
0,1,0,0,4,0,0
1,2,0,0,2,0,0


In [11]:
# Release long-lived objects that hold DB/file handles
import gc

for obj_name in ["memory", "registry", "store"]:
    obj = globals().get(obj_name)
    close_fn = getattr(obj, "close", None)
    if callable(close_fn):
        try:
            close_fn()
            print(f"Closed: {obj_name}")
        except Exception as e:
            print(f"Failed to close {obj_name}: {e}")

for obj_name in ["memory", "registry", "store"]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()
print("Released DB/file handles and cleared related globals.")

Closed: memory
Closed: registry
Released DB/file handles and cleared related globals.


## 6. Release DB handles (optional but recommended)

Run this cell after finishing Notebook 04 to release DuckDB file locks without restarting the kernel.

## ✅ Notebook 04 Complete

The dual-track closed loop has run. The system has:
- **Explorer track**: Generated LLM hypotheses guided by RAG context, synthesized expressions
- **Skeleton track**: Retrieved proven skeleton templates, generated field-substituted variants
- **TrackBandit**: Adaptively allocated simulation budget between both tracks
- **SkeletonExtractor**: Deposited qualified/soft-qualified alphas back into SkeletonRegistry
- Filtered near-duplicates (strict novelty for Explorer, field_coverage for Skeleton)
- Reflected on failures and stored insights for next round

**Next**: `05_results_and_novelty_analysis.ipynb` for full analysis with skeleton family trees.

## ✅ Notebook 04 Complete

The closed loop has run. The system has:
- Generated LLM hypotheses guided by RAG context
- Synthesized and locally validated expressions
- Filtered out near-duplicates using novelty scoring
- Submitted to WQB for backtesting
- Reflected on failures and stored insights for next round

**Next**: `05_results_and_novelty_analysis.ipynb` for full analysis.